In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU detected!")

PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
import transformers, datasets, peft, trl, accelerate, bitsandbytes

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 5.22.1
Datasets: 4.8.5
PEFT: 0.21.2
TRL: 1.18.1
Accelerate: 1.15.0
BitsAndBytes: 0.50.2


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", len(tokenizer))
print("Chat template available:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 151665
Chat template available: True


In [ ]:
qa_data = [
    {"question": "What products does the computer shop sell?", "answer": "The shop sells laptops, desktop computers, keyboards, mice, monitors, and other computer accessories."},
    {"question": "What types of laptops are available?", "answer": "The shop offers laptops for students, office users, and gaming purposes."},
    {"question": "Can I buy a desktop computer?", "answer": "Yes, desktop computers are available for personal, office, and gaming use."},
    {"question": "Do you sell computer keyboards?", "answer": "Yes, the shop sells wired and wireless keyboards."},
    {"question": "Are wireless mice available?", "answer": "Yes, both wired and wireless computer mice are available."},
    {"question": "Do you sell monitors?", "answer": "Yes, the shop offers computer monitors in different sizes."},
    {"question": "Can students buy laptops?", "answer": "Yes, students can choose laptops suitable for studying, assignments, and online learning."},
    {"question": "Do you sell gaming laptops?", "answer": "Yes, the shop offers laptops designed for gaming."},
    {"question": "Can I get a laptop for office work?", "answer": "Yes, the shop offers laptops suitable for office applications and daily work."},
    {"question": "Do you provide laptop chargers?", "answer": "Yes, compatible laptop chargers are available."},
    {"question": "Are headphones available?", "answer": "Yes, the shop sells headphones for computer use."},
    {"question": "Do you sell USB flash drives?", "answer": "Yes, USB flash drives are available in different storage capacities."},
    {"question": "Can I buy an external hard drive?", "answer": "Yes, external hard drives are available for additional data storage."},
    {"question": "Do you sell SSDs?", "answer": "Yes, the shop offers solid-state drives for compatible computers."},
    {"question": "Can I upgrade my computer RAM?", "answer": "RAM upgrades are possible depending on the computer model and motherboard compatibility."},
    {"question": "Do you provide computer accessories?", "answer": "Yes, the shop provides keyboards, mice, headphones, cables, and storage devices."},
    {"question": "Is a warranty provided for laptops?", "answer": "Warranty coverage depends on the product and its manufacturer."},
    {"question": "How can I check my product warranty?", "answer": "You can check the warranty details on your purchase receipt or contact customer support."},
    {"question": "Can I return a damaged product?", "answer": "Please contact customer support with your purchase receipt to review the return options."},
    {"question": "What should I do if my laptop stops working?", "answer": "Contact customer support and provide details about the problem and your purchase."},
    {"question": "Do you repair computers?", "answer": "Please contact the shop to confirm available repair services."},
    {"question": "Can I ask for product recommendations?", "answer": "Yes, customer support can help you choose products based on your needs."},
    {"question": "What laptop is suitable for university students?", "answer": "A laptop with sufficient RAM, storage, and battery life is suitable for university studies."},
    {"question": "What computer is suitable for gaming?", "answer": "A gaming computer should have a capable processor, dedicated graphics card, sufficient RAM, and suitable storage."},
    {"question": "Can I connect a wireless keyboard to my laptop?", "answer": "Yes, if your laptop supports the keyboard's wireless connection method."},
    {"question": "How do I protect my computer from viruses?", "answer": "Use trusted security software, keep your operating system updated, and avoid suspicious downloads."},
    {"question": "Why is my computer running slowly?", "answer": "A computer may run slowly because of insufficient memory, limited storage, background applications, or outdated software."},
    {"question": "How can I improve my laptop performance?", "answer": "You can improve performance by managing startup applications, updating software, and checking memory and storage usage."},
    {"question": "Do you sell computer cables?", "answer": "Yes, the shop offers common computer cables such as HDMI and USB cables."},
    {"question": "How can I contact customer support?", "answer": "You can contact the shop's customer support team for product and service-related questions."}
]

print("Total Q&A pairs:", len(qa_data))

Total Q&A pairs: 30


In [ ]:
import pandas as pd

df = pd.DataFrame(qa_data)

display(df.head(5))

print("Total records:", len(df))
print("Missing values:", df.isnull().sum().sum())
print("Duplicate questions:", df["question"].duplicated().sum())

,question,answer
0,What products does the computer shop sell?,"The shop sells laptops, desktop computers, key..."
1,What types of laptops are available?,"The shop offers laptops for students, office u..."
2,Can I buy a desktop computer?,"Yes, desktop computers are available for perso..."
3,Do you sell computer keyboards?,"Yes, the shop sells wired and wireless keyboards."
4,Are wireless mice available?,"Yes, both wired and wireless computer mice are..."


Total records: 30
Missing values: 0
Duplicate questions: 0


In [ ]:
from datasets import Dataset

formatted_data = []

for item in qa_data:
    formatted_data.append({
        "messages": [
            {"role": "user", "content": item["question"]},
            {"role": "assistant", "content": item["answer"]}
        ]
    })

dataset = Dataset.from_list(formatted_data)

print(dataset)
print("\nFirst training example:")
print(dataset[0])

Dataset({
    features: ['messages'],
    num_rows: 30
})

First training example:
{'messages': [{'content': 'What products does the computer shop sell?', 'role': 'user'}, {'content': 'The shop sells laptops, desktop computers, keyboards, mice, monitors, and other computer accessories.', 'role': 'assistant'}]}


In [ ]:
import json

with open("computer_shop_dataset.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, indent=4, ensure_ascii=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    formatted_data,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

Training samples: 24
Testing samples: 6


In [ ]:
from google.colab import files
files.download("computer_shop_dataset.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
evaluation_data = [
    {
        "question": "What should I check before upgrading my laptop RAM?",
        "answer": "Check the supported RAM type, maximum capacity, speed, and available memory slots for your laptop model."
    },
    {
        "question": "How can I connect my laptop to an external monitor?",
        "answer": "You can connect your laptop to an external monitor using a compatible HDMI or other supported display connection."
    },
    {
        "question": "What should I consider when choosing a laptop for video meetings?",
        "answer": "Consider the webcam quality, microphone, battery life, performance, and internet connectivity."
    },
    {
        "question": "What can I do if my laptop storage is almost full?",
        "answer": "Remove unnecessary files, uninstall unused applications, move files to external storage, or consider a compatible storage upgrade."
    },
    {
        "question": "How can I take care of my laptop battery?",
        "answer": "Use a suitable charger, avoid extreme temperatures, and follow the battery care recommendations provided by the manufacturer."
    },
    {
        "question": "What should I do if I lose my purchase receipt?",
        "answer": "Contact customer support and ask whether they can help verify your purchase using other available purchase details."
    }
]

print("Unseen evaluation questions:", len(evaluation_data))

Unseen evaluation questions: 6


In [ ]:
import json

with open("evaluation_questions.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_data, f, indent=4, ensure_ascii=False)

print("Evaluation dataset saved successfully!")

Evaluation dataset saved successfully!


In [ ]:
from google.colab import files
files.download("evaluation_questions.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from transformers import AutoConfig

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

config = AutoConfig.from_pretrained(model_name)

print("Model:", model_name)
print("Parameters:", config.num_hidden_layers, "layers")
print("Hidden size:", config.hidden_size)
print("GPU:", torch.cuda.get_device_name(0))
print("Available GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Model: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 24 layers
Hidden size: 896
GPU: Tesla T4
Available GPU memory: 14.56 GB


In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit QLoRA configuration created successfully!")
print("Quantization: NF4")
print("Compute dtype: float16")
print("Double quantization: Enabled")

4-bit QLoRA configuration created successfully!
Quantization: NF4
Compute dtype: float16
Double quantization: Enabled


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen model loaded successfully!")
print("Model device:", model.device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!
Model device: cuda:0


In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA training!")

Model prepared for QLoRA training!


In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA configuration created successfully!")
print("Rank:", lora_config.r)
print("Alpha:", lora_config.lora_alpha)
print("Dropout:", lora_config.lora_dropout)

LoRA configuration created successfully!
Rank: 8
Alpha: 16
Dropout: 0.05


In [ ]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(formatted_data)

print("Training dataset:", train_dataset)
print("Training samples:", len(train_dataset))

Training dataset: Dataset({
    features: ['messages'],
    num_rows: 30
})
Training samples: 30


In [ ]:
from trl import SFTTrainer
import inspect

print("SFTTrainer signature:")
print(inspect.signature(SFTTrainer))

SFTTrainer signature:
(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.Tra

In [ ]:
from trl import SFTConfig
import inspect

print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool | None = None, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, to

In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [2]:
import trl
from trl import SFTConfig

print("TRL version:", trl.__version__)
print("SFTConfig imported successfully!")

TRL version: 1.18.1
SFTConfig imported successfully!


In [3]:
from trl import SFTConfig

config = SFTConfig(
    output_dir="./test-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    max_length=512
)

print("SFTConfig created successfully!")
print("Batch size:", config.per_device_train_batch_size)
print("Gradient accumulation:", config.gradient_accumulation_steps)
print("Epochs:", config.num_train_epochs)
print("Learning rate:", config.learning_rate)
print("Max length:", config.max_length)

SFTConfig created successfully!
Batch size: 1
Gradient accumulation: 4
Epochs: 3
Learning rate: 0.0002
Max length: 512


In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("Currently using CPU runtime.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not assigned yet.")

CUDA available: True
GPU: Tesla T4


In [6]:
import torch

gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU Memory:", round(gpu_memory, 2), "GB")
print("CUDA Version:", torch.version.cuda)

GPU: Tesla T4
Total GPU Memory: 14.56 GB
CUDA Version: 13.0


In [7]:
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("All required libraries imported successfully!")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

All required libraries imported successfully!
Transformers: 5.22.1
PEFT: 0.21.2
TRL: 1.18.1
BitsAndBytes: 0.50.2


In [8]:
if "train_dataset" in globals():
    print("Dataset found!")
    print("Number of samples:", len(train_dataset))
    print("Columns:", train_dataset.column_names)
    print("First sample:", train_dataset[0])
else:
    print("Dataset not found. We need to recreate it.")

Dataset not found. We need to recreate it.


In [9]:
from datasets import Dataset

formatted_data = [
    {"messages": [{"role": "user", "content": "What services does TechZone offer?"}, {"role": "assistant", "content": "TechZone is a fictional computer shop offering laptops, desktop computers, accessories, and basic technical support."}]},
    {"messages": [{"role": "user", "content": "Do you sell laptops?"}, {"role": "assistant", "content": "Yes. TechZone offers different laptop models for students, office workers, and general users."}]},
    {"messages": [{"role": "user", "content": "Do you sell desktop computers?"}, {"role": "assistant", "content": "Yes. TechZone offers desktop computers for home, office, and gaming use."}]},
    {"messages": [{"role": "user", "content": "Can I buy a computer mouse?"}, {"role": "assistant", "content": "Yes. Computer mice are among the accessories available at TechZone."}]},
    {"messages": [{"role": "user", "content": "Do you have keyboards?"}, {"role": "assistant", "content": "Yes. TechZone offers keyboards for everyday computing and gaming."}]},
    {"messages": [{"role": "user", "content": "Can I purchase a laptop bag?"}, {"role": "assistant", "content": "Yes. Laptop bags are available as computer accessories."}]},
    {"messages": [{"role": "user", "content": "Do you sell headphones?"}, {"role": "assistant", "content": "Yes. TechZone offers headphones for online meetings, entertainment, and everyday use."}]},
    {"messages": [{"role": "user", "content": "Can I get a monitor?"}, {"role": "assistant", "content": "Yes. External monitors are part of the computer products offered by TechZone."}]},
    {"messages": [{"role": "user", "content": "Do you provide technical support?"}, {"role": "assistant", "content": "Yes. TechZone provides basic technical support for computer-related issues."}]},
    {"messages": [{"role": "user", "content": "Can you help me choose a laptop?"}, {"role": "assistant", "content": "Yes. Tell us your budget and intended use, such as studying, office work, programming, or gaming."}]},
    {"messages": [{"role": "user", "content": "Which laptop is suitable for university students?"}, {"role": "assistant", "content": "A laptop with sufficient RAM, SSD storage, and a suitable processor is generally useful for university work."}]},
    {"messages": [{"role": "user", "content": "What laptop is suitable for programming?"}, {"role": "assistant", "content": "A programming laptop should have a capable processor, sufficient RAM, and SSD storage. The ideal specifications depend on the software you use."}]},
    {"messages": [{"role": "user", "content": "What should I consider before buying a laptop?"}, {"role": "assistant", "content": "Consider the processor, RAM, storage, battery life, display, warranty, and your budget."}]},
    {"messages": [{"role": "user", "content": "What is SSD storage?"}, {"role": "assistant", "content": "SSD stands for Solid State Drive. It is a storage device that generally provides faster data access than a traditional hard disk drive."}]},
    {"messages": [{"role": "user", "content": "How much RAM do I need for everyday tasks?"}, {"role": "assistant", "content": "The required RAM depends on your applications. More RAM helps when running several applications simultaneously."}]},
    {"messages": [{"role": "user", "content": "What is the difference between RAM and storage?"}, {"role": "assistant", "content": "RAM temporarily holds data used by running programs, while storage keeps files and applications even when the computer is turned off."}]},
    {"messages": [{"role": "user", "content": "Do you sell gaming computers?"}, {"role": "assistant", "content": "Yes. TechZone offers computer options intended for gaming. The suitable configuration depends on the games and budget."}]},
    {"messages": [{"role": "user", "content": "Can I upgrade my computer RAM?"}, {"role": "assistant", "content": "RAM upgrades may be possible depending on the computer model and its hardware compatibility."}]},
    {"messages": [{"role": "user", "content": "Can I upgrade my laptop storage?"}, {"role": "assistant", "content": "Storage upgrades may be possible depending on the laptop model and available storage slots."}]},
    {"messages": [{"role": "user", "content": "Why is my computer running slowly?"}, {"role": "assistant", "content": "A computer may run slowly because of limited RAM, insufficient storage space, background applications, or software issues."}]},
    {"messages": [{"role": "user", "content": "What should I do if my laptop does not turn on?"}, {"role": "assistant", "content": "Check the power adapter, charging connection, and battery. If the problem continues, seek technical assistance."}]},
    {"messages": [{"role": "user", "content": "How can I protect my computer from viruses?"}, {"role": "assistant", "content": "Use trusted security software, keep your operating system updated, and avoid suspicious links and downloads."}]},
    {"messages": [{"role": "user", "content": "Do you provide computer maintenance?"}, {"role": "assistant", "content": "TechZone provides basic computer maintenance support. Contact the shop to confirm the available services."}]},
    {"messages": [{"role": "user", "content": "Can I ask about product availability?"}, {"role": "assistant", "content": "Yes. Contact TechZone with the product name or model to check its availability."}]},
    {"messages": [{"role": "user", "content": "How can I compare two laptops?"}, {"role": "assistant", "content": "Compare their processors, RAM, storage, graphics, display quality, battery life, warranty, and prices."}]},
    {"messages": [{"role": "user", "content": "What is a graphics card?"}, {"role": "assistant", "content": "A graphics card processes visual information and helps display images, videos, and 3D graphics."}]},
    {"messages": [{"role": "user", "content": "Is a dedicated graphics card necessary?"}, {"role": "assistant", "content": "A dedicated graphics card is useful for gaming, 3D work, and some machine learning tasks, but is not essential for basic office work."}]},
    {"messages": [{"role": "user", "content": "Do you sell computer cables?"}, {"role": "assistant", "content": "Yes. TechZone offers computer-related cables and connectivity accessories."}]},
    {"messages": [{"role": "user", "content": "How can I contact TechZone?"}, {"role": "assistant", "content": "You can contact the shop through its official communication channels to ask about products and services."}]},
    {"messages": [{"role": "user", "content": "What information should I provide when requesting technical support?"}, {"role": "assistant", "content": "Provide your device model, operating system, description of the problem, and any error messages displayed."}]}
]

train_dataset = Dataset.from_list(formatted_data)

print("Dataset created successfully!")
print("Number of samples:", len(train_dataset))
print("Columns:", train_dataset.column_names)
print("First sample:", train_dataset[0])

Dataset created successfully!
Number of samples: 30
Columns: ['messages']
First sample: {'messages': [{'content': 'What services does TechZone offer?', 'role': 'user'}, {'content': 'TechZone is a fictional computer shop offering laptops, desktop computers, accessories, and basic technical support.', 'role': 'assistant'}]}


In [10]:
print("Dataset sample:")

for message in train_dataset[0]["messages"]:
    print(message["role"].upper() + ":")
    print(message["content"])
    print()

Dataset sample:
USER:
What services does TechZone offer?

ASSISTANT:
TechZone is a fictional computer shop offering laptops, desktop computers, accessories, and basic technical support.



In [11]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Chat template available:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!
EOS token: <|im_end|>
PAD token: <|endoftext|>
Chat template available: True


In [12]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

formatted_dataset = train_dataset.map(format_chat)

print("Dataset formatted successfully!")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Dataset formatted successfully!
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What services does TechZone offer?<|im_end|>
<|im_start|>assistant
TechZone is a fictional computer shop offering laptops, desktop computers, accessories, and basic technical support.<|im_end|>



In [13]:
split_dataset = formatted_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 24
Testing samples: 6


In [14]:
import inspect
from trl import SFTTrainer

print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None =

In [15]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configured successfully!")
print("Quantization type: NF4")
print("Compute dtype: FP16")

4-bit quantization configured successfully!
Quantization type: NF4
Compute dtype: FP16


In [16]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen model loaded successfully!")
print("Model device:", model.device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!
Model device: cuda:0


In [17]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA training!")

Model prepared for QLoRA training!


In [18]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("LoRA configuration created successfully!")
print("Rank:", peft_config.r)
print("Alpha:", peft_config.lora_alpha)
print("Dropout:", peft_config.lora_dropout)
print("Target modules:", peft_config.target_modules)

LoRA configuration created successfully!
Rank: 8
Alpha: 16
Dropout: 0.05
Target modules: {'q_proj', 'v_proj', 'o_proj', 'k_proj'}


In [19]:
from peft import get_peft_model

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [20]:
print("GPU Memory Allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("GPU Memory Reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

GPU Memory Allocated: 0.68 GB
GPU Memory Reserved: 0.96 GB


In [21]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./techzone-qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    max_length=512,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    dataset_text_field="text"
)

print("Training configuration created successfully!")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)

Training configuration created successfully!
Epochs: 3
Batch size: 1
Learning rate: 0.0002
FP16: True


In [22]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

print("SFTTrainer created successfully!")

Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

SFTTrainer created successfully!


In [24]:
print("GPU:", torch.cuda.get_device_name(0))
print("GPU capability:", torch.cuda.get_device_capability(0))

print("\nTraining precision:")
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

print("\nTrainable parameter data types:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, ":", param.dtype)
        break

print("\nModel configuration dtype:", model.config.torch_dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


GPU: Tesla T4
GPU capability: (7, 5)

Training precision:
FP16: True
BF16: False

Trainable parameter data types:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight : torch.bfloat16

Model configuration dtype: torch.bfloat16


In [25]:
import gc
import torch
from transformers import AutoModelForCausalLM

del trainer, model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("Model reloaded successfully!")
print("Model dtype:", model.config.torch_dtype)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model reloaded successfully!
Model dtype: torch.float16


In [26]:
from peft import prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [27]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

print("SFTTrainer recreated successfully!")

Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

SFTTrainer recreated successfully!


In [29]:
print("Trainable parameter data types:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, ":", param.dtype)

print("\nTrainer FP16:", trainer.args.fp16)
print("Trainer BF16:", trainer.args.bf16)
print("Model dtype:", model.config.torch_dtype)

Trainable parameter data types:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight : torch.bfloat16
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight : torch.bfloat16
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight : torch.bfloat16
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight : torch.bfloat16
base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight : torch.bfloat16


In [30]:
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

print("LoRA adapter weights converted to FP16!")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, ":", param.dtype)
        break

LoRA adapter weights converted to FP16!
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight : torch.float16


In [31]:
dtypes = set(
    param.dtype
    for param in model.parameters()
    if param.requires_grad
)

print("Trainable parameter dtypes:", dtypes)
print("FP16 training:", trainer.args.fp16)
print("BF16 training:", trainer.args.bf16)

Trainable parameter dtypes: {torch.float16}
FP16 training: True
BF16 training: False


In [32]:
import gc

del trainer
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

print("SFTTrainer recreated successfully!")

Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

SFTTrainer recreated successfully!


In [33]:
train_result = trainer.train()

NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [34]:
from collections import Counter

grad_dtypes = Counter(
    str(param.grad.dtype)
    for param in model.parameters()
    if param.grad is not None
)

print("Gradient dtypes:", grad_dtypes)

param_dtypes = Counter(
    str(param.dtype)
    for param in model.parameters()
    if param.requires_grad
)

print("Trainable parameter dtypes:", param_dtypes)

Gradient dtypes: Counter({'torch.bfloat16': 192})
Trainable parameter dtypes: Counter({'torch.bfloat16': 192})


In [35]:
import gc
import torch

del trainer
gc.collect()
torch.cuda.empty_cache()

# Convert trainable LoRA parameters to FP32
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

# Disable mixed-precision training
training_args.fp16 = False
training_args.bf16 = False

from collections import Counter

print("Trainable dtypes:", Counter(
    str(p.dtype) for p in model.parameters() if p.requires_grad
))
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

Trainable dtypes: Counter({'torch.float32': 192})
FP16: False
BF16: False


In [36]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

print("SFTTrainer recreated successfully!")

Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

SFTTrainer recreated successfully!


In [37]:
print("Trainable dtypes:", {
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
})

print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)
print("GPU:", torch.cuda.get_device_name(0))

Trainable dtypes: {'torch.bfloat16'}
FP16: False
BF16: False
GPU: Tesla T4


In [38]:
print("Model dtype:", model.dtype)
print("Model config dtype:", model.config.torch_dtype)
print("4-bit compute dtype:", bnb_config.bnb_4bit_compute_dtype)
print("LoRA config:", model.peft_config["default"])

print("\nTrainable parameter dtypes:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

Model dtype: torch.float32
Model config dtype: torch.float16
4-bit compute dtype: torch.float16
LoRA config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path='Qwen/Qwen2.5-0.5B-Instruct', revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj', 'o_proj', 'k_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False,

In [39]:
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

from collections import Counter

print("Trainer LoRA dtypes:", Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))

Trainer LoRA dtypes: Counter({'torch.float32': 192})


In [40]:
train_result = trainer.train()

Step,Training Loss
1,3.813456
2,4.490630
3,4.047939
4,3.593917
5,3.783630
6,3.338271
7,2.939119
8,3.104895
9,3.090172
10,2.889123


In [41]:
print(train_result)
print("\nTraining metrics:")
print(train_result.metrics)

TrainOutput(global_step=18, training_loss=3.0261338816748724, metrics={'train_runtime': 38.9392, 'train_samples_per_second': 1.849, 'train_steps_per_second': 0.462, 'total_flos': 8729662411008.0, 'train_loss': 3.0261338816748724, 'epoch': 3.0})

Training metrics:
{'train_runtime': 38.9392, 'train_samples_per_second': 1.849, 'train_steps_per_second': 0.462, 'total_flos': 8729662411008.0, 'train_loss': 3.0261338816748724, 'epoch': 3.0}


In [42]:
save_path = "./techzone-qlora/final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Fine-tuned model saved successfully!")

Fine-tuned model saved successfully!


In [43]:
eval_results = trainer.evaluate()

print("Evaluation Results:")
print(eval_results)

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
2.472675,2.382323,18,2.052581,0.627628,4053.000000


Evaluation Results:
{'eval_loss': 2.3823232650756836, 'eval_entropy': 2.0525805950164795, 'eval_mean_token_accuracy': 0.6276276111602783, 'eval_num_tokens': 4053.0}


In [44]:
test_questions = []

for item in test_dataset:
    question = next(
        msg["content"]
        for msg in item["messages"]
        if msg["role"] == "user"
    )
    test_questions.append(question)

for i, question in enumerate(test_questions, 1):
    print(f"Question {i}: {question}")

Question 1: What information should I provide when requesting technical support?
Question 2: Do you sell gaming computers?
Question 3: Can I get a monitor?
Question 4: What is a graphics card?
Question 5: How can I compare two laptops?
Question 6: Do you provide computer maintenance?


In [46]:
model.eval()

question = test_questions[0]

messages = [
    {"role": "system", "content": test_dataset[0]["messages"][0]["content"]},
    {"role": "user", "content": question}
]

# Prepare input correctly
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)["input_ids"].to(model.device)

# Original model response
with torch.no_grad():
    with model.disable_adapter():
        original_output = model.generate(
            input_ids=inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

# Fine-tuned model response
with torch.no_grad():
    finetuned_output = model.generate(
        input_ids=inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

original_answer = tokenizer.decode(
    original_output[0][inputs.shape[1]:],
    skip_special_tokens=True
)

finetuned_answer = tokenizer.decode(
    finetuned_output[0][inputs.shape[1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("\nOriginal Model Answer:\n", original_answer)
print("\nFine-Tuned Model Answer:\n", finetuned_answer)

Question: What information should I provide when requesting technical support?

Original Model Answer:
 When requesting technical support, you should provide the following information:

  * Your name and contact details (phone number, email address)
  * A brief description of your problem or issue
  * Any relevant screenshots or videos to help us diagnose the problem
  * Any previous attempts at solving the problem
  * Any troubleshooting steps that have been taken

It's also helpful to include any error messages or logs from the system that may be helpful in diagnosing the problem.

Fine-Tuned Model Answer:
 When requesting technical support, you should provide the following information:

  * Your name and contact details (phone number, email address)
  * A brief description of your problem or issue
  * Any relevant screenshots or videos to help us diagnose the problem
  * Any previous solutions or steps that have been taken to resolve the issue
  * Any other relevant information that

In [47]:
for i in range(1, len(test_questions)):
    question = test_questions[i]

    messages = [
        {"role": "system", "content": test_dataset[i]["messages"][0]["content"]},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )["input_ids"].to(model.device)

    with torch.no_grad():
        with model.disable_adapter():
            original_output = model.generate(
                input_ids=inputs,
                max_new_tokens=120,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        finetuned_output = model.generate(
            input_ids=inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    original_answer = tokenizer.decode(
        original_output[0][inputs.shape[1]:],
        skip_special_tokens=True
    )

    finetuned_answer = tokenizer.decode(
        finetuned_output[0][inputs.shape[1]:],
        skip_special_tokens=True
    )

    print(f"\n{'='*60}")
    print(f"Question {i+1}: {question}")
    print("\nOriginal Model:\n", original_answer)
    print("\nFine-Tuned Model:\n", finetuned_answer)


Question 2: Do you sell gaming computers?

Original Model:
 Yes, I do sell a variety of products including gaming computers. However, it's important to note that the specific offerings and locations may vary depending on your location or region. If you're interested in purchasing a gaming computer, I can provide more detailed information about what we offer. Would you like assistance with any other questions related to gaming technology?

Fine-Tuned Model:
 Yes, I do.

Question 3: Can I get a monitor?

Original Model:
 Yes, you can get a monitor. A monitor is a device that displays images and text on a screen, typically in a room or other enclosed space. It's commonly used in homes, offices, and other settings where people need to see information while working. If you're looking for a specific brand or model of monitor, please let me know the specifications so we can find the right one for you.

Fine-Tuned Model:
 Yes, you can get a monitor.

Question 4: What is a graphics card?

Orig

In [48]:
import pandas as pd

comparison_data = {
    "Question": [
        "Technical support",
        "Gaming computers",
        "Monitor availability",
        "Graphics card",
        "Laptop comparison",
        "Computer maintenance"
    ],
    "Original Model": [
        "Detailed and relevant",
        "Detailed response",
        "Extra explanation",
        "Long technical explanation",
        "Relevant comparison criteria",
        "General service response"
    ],
    "Fine-Tuned Model": [
        "Detailed and relevant",
        "Very short response",
        "Direct response",
        "Concise technical definition",
        "Some irrelevant criteria",
        "Detailed service examples"
    ]
}

comparison_df = pd.DataFrame(comparison_data)

display(comparison_df)

,Question,Original Model,Fine-Tuned Model
0,Technical support,Detailed and relevant,Detailed and relevant
1,Gaming computers,Detailed response,Very short response
2,Monitor availability,Extra explanation,Direct response
3,Graphics card,Long technical explanation,Concise technical definition
4,Laptop comparison,Relevant comparison criteria,Some irrelevant criteria
5,Computer maintenance,General service response,Detailed service examples


In [49]:
comparison_df.insert(
    0,
    "Question No.",
    range(1, len(comparison_df) + 1)
)

comparison_df = comparison_df.drop(columns=["Question"])

display(comparison_df)

,Question No.,Original Model,Fine-Tuned Model
0,1,Detailed and relevant,Detailed and relevant
1,2,Detailed response,Very short response
2,3,Extra explanation,Direct response
3,4,Long technical explanation,Concise technical definition
4,5,Relevant comparison criteria,Some irrelevant criteria
5,6,General service response,Detailed service examples


In [50]:
comparison_df = pd.DataFrame({
    "Question No.": [1, 2, 3, 4, 5, 6],
    "Original Model": [
        "Detailed and relevant",
        "Detailed response",
        "Extra explanation",
        "Long technical explanation",
        "Relevant comparison criteria",
        "General service response"
    ],
    "Fine-Tuned Model": [
        "Detailed and relevant",
        "Very short response",
        "Direct response",
        "Concise technical definition",
        "Some irrelevant criteria",
        "Detailed service examples"
    ]
})

display(comparison_df)

,Question No.,Original Model,Fine-Tuned Model
0,1,Detailed and relevant,Detailed and relevant
1,2,Detailed response,Very short response
2,3,Extra explanation,Direct response
3,4,Long technical explanation,Concise technical definition
4,5,Relevant comparison criteria,Some irrelevant criteria
5,6,General service response,Detailed service examples


In [51]:
display(comparison_df.style.hide(axis="index"))

Question No.,Original Model,Fine-Tuned Model
1,Detailed and relevant,Detailed and relevant
2,Detailed response,Very short response
3,Extra explanation,Direct response
4,Long technical explanation,Concise technical definition
5,Relevant comparison criteria,Some irrelevant criteria
6,General service response,Detailed service examples


In [52]:
comparison_df.to_csv(
    "techzone_model_comparison.csv",
    index=False
)

print("Comparison results saved successfully!")

Comparison results saved successfully!


In [53]:
unseen_questions = [
    "My laptop overheats and shuts down. What should I do?",
    "What should I consider when buying a laptop for university studies?",
    "What is the difference between an SSD and an HDD?",
    "What information do you need to recommend a desktop computer?",
    "My computer is running slowly. What basic steps can I try?"
]

known_questions = {
    message["content"].strip().lower()
    for row in formatted_dataset
    for message in row["messages"]
    if message["role"] == "user"
}

overlap = [
    q for q in unseen_questions
    if q.strip().lower() in known_questions
]

print("Number of unseen questions:", len(unseen_questions))
print("Exact matches in original dataset:", overlap)

Number of unseen questions: 5
Exact matches in original dataset: []


In [54]:
model.eval()

unseen_comparison = []

for i, question in enumerate(unseen_questions, start=1):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant for TechZone, a computer shop."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )["input_ids"].to(model.device)

    with torch.no_grad():

        with model.disable_adapter():
            original_output = model.generate(
                input_ids=inputs,
                max_new_tokens=120,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        finetuned_output = model.generate(
            input_ids=inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    original_answer = tokenizer.decode(
        original_output[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    finetuned_answer = tokenizer.decode(
        finetuned_output[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    unseen_comparison.append({
        "Question": question,
        "Original Model": original_answer,
        "Fine-Tuned Model": finetuned_answer
    })

    print(f"Question {i}: {question}")
    print("\nOriginal:", original_answer)
    print("\nFine-Tuned:", finetuned_answer)
    print("-" * 80)

Question 1: My laptop overheats and shuts down. What should I do?

Original: If your laptop overheats and shuts down, the first step is to check if there is any damage or malfunction with the laptop. You can do this by checking the laptop's case, which may have a temperature sensor that indicates whether the laptop is overheating.

Once you have identified any potential issues, you can take the following steps:

1. Check the battery: If the laptop has a battery, make sure it is fully charged and not running out of power.
2. Check the fans: Make sure all fans on the laptop are working properly and not clogged up.
3. Check the cooling

Fine-Tuned: If your laptop overheats and shuts down, the first step is to check if there is any problem with the hardware of the laptop. If you find that the laptop has a faulty component or a broken circuit, it may be necessary to replace the laptop with a new one.
--------------------------------------------------------------------------------
Question 2

In [55]:
unseen_df = pd.DataFrame(unseen_comparison)

unseen_df.to_csv(
    "techzone_unseen_comparison.csv",
    index=False
)

print("Unseen question comparison saved successfully!")

Unseen question comparison saved successfully!
